# Pregenerated Outputs LMDB Dataset Explorer

Explores LMDB datasets written by `DiskRewardLogger` (reward logging for RL training runs).
Point `LMDB_PATH` at the database to load and browse.

In [ ]:
import html
import typing

import IPython.display as ipy_display
import pandas as pd

import pyine.data.utils.lmdb_io as lmdb_io
import pyine.utils.notebooks as nb_utils
import pyine.utils.reprod

In [ ]:
pyine.utils.reprod.entrypoint_setup()
nb_utils.setup_notebook_plotting(use_seaborn=True, seaborn_style="whitegrid")

# ------------ CONFIGURATION ------------
LMDB_PATH = "../generation_export.lmdb"  # path to the LMDB directory written by DiskRewardLogger

# optional: filter records by key prefix pattern (fnmatch syntax, e.g., "train/*")
KEY_PATTERN: str | None = None  # None = load all records

In [ ]:
# load all records into memory, then close the reader to release the LMDB handle
reader = lmdb_io.LMDBReader(LMDB_PATH)
metadata = reader.get_metadata()
print(f"Database: {reader.path}")
print(f"Total records: {len(reader)}")
print(f"Serialization: {metadata.get('serialization')}")
print(f"Size on disk: {reader.get_size_on_disk() / (1024**2):.1f} MB")

if KEY_PATTERN is not None:
    indices, keys = reader.get_indices(KEY_PATTERN, return_keys=True)  # type: ignore[assignment]
    print(f"Matched {len(indices)} records for pattern '{KEY_PATTERN}'")
    records = [reader.get(idx) for idx in indices]
    for record, key in zip(records, keys, strict=False):
        record["_key"] = key
else:
    keys = list(reader.key_map.keys())
    records = list(reader)
    for record, key in zip(records, keys, strict=False):
        record["_key"] = key

reader.close()
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records into DataFrame with columns: {list(df.columns)}")

## Overview & Summary Statistics

In [ ]:
# summary statistics for numeric fields
print("=== Reward Statistics ===")
if "reward_total" in df.columns:
    reward_stats = df["reward_total"].describe()
    print(reward_stats.to_string())
else:
    print("(no reward_total column found)")

print("\n=== Key Prefix Distribution ===")
if "key_prefix" in df.columns:
    print(df["key_prefix"].value_counts().to_string())

print("\n=== Step Range ===")
if "step" in df.columns and df["step"].notna().any():
    print(f"  min={df['step'].min()}, max={df['step'].max()}, unique={df['step'].nunique()}")

print("\n=== Predict Type Distribution ===")
if "predict_type" in df.columns:
    print(df["predict_type"].value_counts(dropna=False).to_string())

print("\n=== Code Type Distribution ===")
if "code_type" in df.columns:
    print(df["code_type"].value_counts(dropna=False).to_string())

## Sample Browser

In [ ]:
def display_sample(row: pd.Series) -> None:
    """Display a single sample record with HTML formatting."""
    reasoning = row.get("reasoning")
    has_reasoning = bool(reasoning) if isinstance(reasoning, str) else False
    final_answer = row.get("final_answer")
    has_answer = bool(final_answer) if isinstance(final_answer, str) else False
    # status indicators
    status_parts: list[str] = []
    if not has_reasoning:
        status_parts.append('<span style="color: orange; font-weight: bold;">NO REASONING</span>')
    if not has_answer:
        status_parts.append('<span style="color: orange; font-weight: bold;">NO ANSWER</span>')
    status_html = " | ".join(status_parts) if status_parts else '<span style="color: green;">OK</span>'
    parts: list[str] = []
    key = row.get("_key", "N/A")
    parts.append(f"<h4>{html.escape(str(key))} [{status_html}]</h4>")
    parts.append("<table style='border-collapse: collapse; margin-bottom: 10px;'>")

    # metadata rows
    def _row(label: str, value: typing.Any) -> None:
        if value is not None and not (isinstance(value, float) and pd.isna(value)):
            parts.append(
                f"<tr><td style='padding-right:12px'><b>{label}:</b></td><td>{html.escape(str(value))}</td></tr>"
            )

    _row("Reward", f"{row.get('reward_total'):.4f}" if pd.notna(row.get("reward_total")) else None)
    _row("Step", int(row["step"]) if pd.notna(row.get("step")) else None)
    _row("Batch", int(row["batch_count"]) if pd.notna(row.get("batch_count")) else None)
    _row("Rank", int(row["rank"]) if pd.notna(row.get("rank")) else None)
    _row("Predict Type", row.get("predict_type"))
    _row("Code Type", row.get("code_type"))
    _row("Categories", row.get("categories"))
    _row("Tags", row.get("tags"))
    _row(
        "Difficulty",
        f"{row.get('difficulty_score'):.3f} (bin={row.get('difficulty_bin')})"
        if pd.notna(row.get("difficulty_score"))
        else None,
    )
    # reward terms
    terms = row.get("reward_terms")
    if terms and isinstance(terms, dict):
        terms_str = ", ".join(f"{k}={v:.4f}" for k, v in terms.items())
        _row("Terms", terms_str)
    parts.append("</table>")
    # text fields in scrollable blocks
    block_style = (
        "max-height:200px; overflow-y:auto; background:#f5f5f5; padding:8px; "
        "font-family:monospace; font-size:12px; white-space:pre-wrap; "
        "border:1px solid #ddd; margin-bottom:8px;"
    )
    for field_name in ("prompt", "reasoning", "final_answer", "expected_output", "model_output"):
        value = row.get(field_name)
        if value and isinstance(value, str):
            parts.append(f"<details><summary><b>{field_name}</b> ({len(value)} chars)</summary>")
            parts.append(f"<div style='{block_style}'>{html.escape(value)}</div></details>")
    ipy_display.display(ipy_display.HTML("".join(parts)))


try:
    import ipywidgets

    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not available - interactive browser disabled")


def create_sample_browser(
    samples_df: pd.DataFrame,
) -> "ipywidgets.VBox":
    """Create an interactive sample browser widget."""
    current_df = samples_df.copy()
    # filter controls — checkboxes
    show_missing_answer = ipywidgets.Checkbox(value=False, description="Missing answer")
    show_missing_reasoning = ipywidgets.Checkbox(value=False, description="Missing reasoning")
    # filter controls — dropdowns
    all_label = "(all)"
    prefix_values = sorted(samples_df["key_prefix"].dropna().unique()) if "key_prefix" in samples_df.columns else []
    prefix_dropdown = ipywidgets.Dropdown(
        options=[all_label] + list(prefix_values),
        value=all_label,
        description="Prefix:",
    )
    code_types = sorted(samples_df["code_type"].dropna().unique()) if "code_type" in samples_df.columns else []
    code_dropdown = ipywidgets.Dropdown(
        options=[all_label] + list(code_types),
        value=all_label,
        description="Code type:",
    )
    # navigation
    idx_slider = ipywidgets.IntSlider(value=0, min=0, max=max(0, len(current_df) - 1), description="Index:")
    prev_btn = ipywidgets.Button(description="< Prev")
    next_btn = ipywidgets.Button(description="Next >")
    # output area
    output = ipywidgets.Output()
    info_label = ipywidgets.HTML(value=f"Total samples: {len(current_df)}")

    def apply_filters() -> None:
        nonlocal current_df
        filtered = samples_df.copy()
        if show_missing_answer.value:
            filtered = filtered[filtered["final_answer"].apply(lambda x: not x if isinstance(x, str) else True)]
        if show_missing_reasoning.value:
            filtered = filtered[filtered["reasoning"].apply(lambda x: not x if isinstance(x, str) else True)]
        if prefix_dropdown.value != all_label and "key_prefix" in filtered.columns:
            filtered = filtered[filtered["key_prefix"] == prefix_dropdown.value]
        if code_dropdown.value != all_label and "code_type" in filtered.columns:
            filtered = filtered[filtered["code_type"] == code_dropdown.value]
        current_df = filtered
        idx_slider.max = max(0, len(current_df) - 1)
        idx_slider.value = min(idx_slider.value, idx_slider.max)
        info_label.value = f"Showing {len(current_df)} of {len(samples_df)} samples"
        update_display(None)

    def update_display(_: typing.Any) -> None:
        output.clear_output()
        with output:
            if len(current_df) == 0:
                print("No samples match current filters")
                return
            idx = idx_slider.value
            if idx < len(current_df):
                display_sample(current_df.iloc[idx])

    def on_prev(_: typing.Any) -> None:
        if idx_slider.value > 0:
            idx_slider.value -= 1

    def on_next(_: typing.Any) -> None:
        if idx_slider.value < idx_slider.max:
            idx_slider.value += 1

    # connect handlers
    idx_slider.observe(update_display, names="value")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    show_missing_answer.observe(lambda _: apply_filters(), names="value")
    show_missing_reasoning.observe(lambda _: apply_filters(), names="value")
    prefix_dropdown.observe(lambda _: apply_filters(), names="value")
    code_dropdown.observe(lambda _: apply_filters(), names="value")
    # initial display
    update_display(None)
    # layout
    checkbox_row = ipywidgets.HBox([show_missing_answer, show_missing_reasoning])
    dropdown_row = ipywidgets.HBox([prefix_dropdown, code_dropdown])
    nav_box = ipywidgets.HBox([prev_btn, idx_slider, next_btn])
    return ipywidgets.VBox([info_label, checkbox_row, dropdown_row, nav_box, output])


if WIDGETS_AVAILABLE and len(df) > 0:
    browser = create_sample_browser(df)
    ipy_display.display(browser)
else:
    print("No samples to browse" if len(df) == 0 else "Widgets unavailable; use display_sample(df.iloc[N]) manually")